# Run Backtest

Quick notebook for testing backtests. Two paths:
- **Preset** — load a named YAML config from `presets/`
- **Custom strategy** — pass a Python strategy instance directly

In [1]:
%load_ext autoreload
%autoreload 2

import os
# os.environ["GNOME_ROOT"] = "/path/to/gnome"
# os.environ["GNOME_REGISTRY_API_KEY"] = "..."

from gnomepy.java._jvm import ensure_jvm_started
ensure_jvm_started()

## Option A — run a preset

In [2]:
from datetime import datetime
from gnomepy_research.presets import load_preset, list_presets

print("Available presets:", list_presets())

backtest = load_preset(
    "mm_btc_30m",
    start_date=datetime(2026, 1, 23, 10, 30),
    end_date=datetime(2026, 1, 23, 13, 0),
)

results = backtest.run()

Available presets: ['mm_btc_30m', 'momentum_btc_30m']


SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
Apr 15, 2026 8:09:22 PM group.gnometrading.backtest.driver.BacktestDriver executeUntil
SEVERE: Exception at timestamp 1769164201964500000: Received execution report for unknown order with clientOidCounter: 1


java.lang.RuntimeException: java.lang.RuntimeException: java.lang.IllegalStateException: Received execution report for unknown order with clientOidCounter: 1

## Option B — run with a custom Python strategy

Pass an instantiated strategy to override the one in the preset, or build a `BacktestConfig` directly.

In [ ]:
from gnomepy_research.presets import load_preset
from gnomepy_research.strategies.momentum import MomentumTaker

strategy = MomentumTaker(
    exchange_id=1,
    security_id=1,
    size=100_000,
    max_position=5,
    threshold_bps=3.0,
)

backtest = load_preset(
    "momentum_btc_30m",
    strategy=strategy,
    start_date=datetime(2026, 1, 23, 10, 30),
    end_date=datetime(2026, 1, 23, 13, 0),
)

results = backtest.run()

## Inspect results

In [ ]:
from gnomepy_research.reporting import BacktestReport

report = BacktestReport(results, backtest=backtest)
report

In [ ]:
report.summary_df()

In [ ]:
report.plot_pnl().show()

In [ ]:
report.plot_position().show()

In [ ]:
report.save_html("report.html")
# ! open report.html